In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Eigene Module laden
from FIT_python.pipeline_individual_id.geometric_pairwise_projection import generate_pairwise_comparisons_from_df
from  FIT_python.pipeline_individual_id.geometric_pairwise_projection  import run_all_pairwise_projections

# Pfade definieren
data_path = Path("C:/Users/Frede/OneDrive/Projects and Disschaptors/FIT_package/data/splits/eurasian_otter_(1)")
csv_path = data_path / "train.csv"

# Daten einlesen
df = pd.read_csv(csv_path)

# Nur numerische Spaltennamen mit 'dist', 'ang' oder 't' am Anfang
feature_cols = [
    col for col in df.columns
    if col.startswith(("dist", "ang", "t")) and pd.api.types.is_numeric_dtype(df[col])
]
print(f"{len(feature_cols)} numerische Feature-Spalten gefunden.")


205 numerische Feature-Spalten gefunden.


In [2]:
from  FIT_python.pipeline_individual_id.geometric_pairwise_projection  import run_all_pairwise_projections
from FIT_python.pipeline_individual_id.geometric_pairwise_projection import  run_all_pairwise_projections_parallel
from FIT_python.pipeline_individual_id.geometric_pairwise_projection import generate_pairwise_comparisons_from_df


# Vergleiche generieren
comparisons = generate_pairwise_comparisons_from_df(
    df,
    group_sizes=[5, 7],
    n_repeats=2,
    mode="both",
    selfmatch_factor=1
)

# Parallele Projektionen mit Fortschrittsbalken
results = run_all_pairwise_projections_parallel(
    comparisons=comparisons,
    df=df,
    feature_cols=feature_cols,
    k_features=15,
    reducer="lda",
    selection_method="forward",
    n_components=2,
    debug=False,  # debug=True macht Output unübersichtlich
    n_jobs=-1     # nutze alle Kerne
)


# In DataFrame umwandeln
results_df = pd.DataFrame(results)
results_df.head()


100%|██████████| 2620/2620 [02:48<00:00, 15.53it/s]


,ind_a,ind_b,same_individual,selected_features,selection_method,center_distance_ab,center_distance_rcv_a,center_distance_rcv_b,comparison_id
0,zanetka,zewe,False,"[dist2_13, dist10_11, t6_7_17, dist6_8, ang4_9...",forward,2.241741,2.399697,2.073357,0
1,zanetka,zewe,False,"[dist1_13, dist2_17, ang1_3_4, dist9_15, dist2...",forward,2.728696,2.118021,2.315993,1
2,sam,zanetka,False,"[dist2_13, dist10_11, ang2_1_4, dist1_9, dist8...",forward,3.504994,2.227404,2.492561,2
3,sam,zanetka,False,"[dist15_17, dist7_17, dist3_13, dist9_16, ang4...",forward,4.202524,3.177300,2.413014,3
4,sam,shorfy,False,"[dist6_10, dist5_15, dist1_11, dist1_9, dist2_...",forward,4.358537,2.955875,2.414069,4


In [3]:
import os

output_path = r"C:\Users\Frede\OneDrive\Projects and Disschaptors\FIT_package\results\data"
os.makedirs(output_path, exist_ok=True)

results_df.to_csv(os.path.join(output_path, "results_pairwise_pca_forward.csv"), index=False)
print("Datei erfolgreich gespeichert.")



Datei erfolgreich gespeichert.


In [ ]:
from tqdm import tqdm
import pandas as pd
import os

# Parameter
feature_counts = [5, 10, 15, 20]
reducers = ["pca", "lda"]
selection_method = "forward"
n_components = 2
n_jobs = -1

# Ausgabeordner
output_dir = r"C:\Users\Frede\OneDrive\Projects and Disschaptors\FIT_package\results\data"
os.makedirs(output_dir, exist_ok=True)

all_batch_results = []

# Schleife über alle Kombinationen aus Feature-Anzahl und Reduktionsmethode
for k in feature_counts:
    for reducer in reducers:
        print(f"Starte run_all_pairwise_projections_parallel mit k={k}, reducer={reducer}")

        results = run_all_pairwise_projections_parallel(
            comparisons=comparisons,
            df=df,
            feature_cols=feature_cols,
            k_features=k,
            reducer=reducer,
            selection_method=selection_method,
            n_components=n_components,
            debug=False,
            n_jobs=n_jobs
        )

        # In DataFrame umwandeln
        results_df = pd.DataFrame(results)
        results_df["k_features"] = k
        results_df["reducer"] = reducer

        # Zwischenspeichern
        filename = f"results_pairwise_{reducer}_forward_k{k}.csv"
        results_df.to_csv(os.path.join(output_dir, filename), index=False)

        # Ergebnisse für spätere Zusammenfassung merken
        all_batch_results.append(results_df)

# Optional: Gesamtergebnis als ein DataFrame
full_results_df = pd.concat(all_batch_results, ignore_index=True)
full_results_df.to_csv(os.path.join(output_dir, "all_results_combined.csv"), index=False)


Starte run_all_pairwise_projections_parallel mit k=5, reducer=pca


100%|██████████| 2620/2620 [00:35<00:00, 73.35it/s]


Starte run_all_pairwise_projections_parallel mit k=5, reducer=lda


100%|██████████| 2620/2620 [00:32<00:00, 80.66it/s]


Starte run_all_pairwise_projections_parallel mit k=10, reducer=pca


100%|██████████| 2620/2620 [01:09<00:00, 37.50it/s]


Starte run_all_pairwise_projections_parallel mit k=10, reducer=lda


100%|██████████| 2620/2620 [01:09<00:00, 37.56it/s]


Starte run_all_pairwise_projections_parallel mit k=15, reducer=pca


 86%|████████▌ | 2256/2620 [01:36<00:17, 20.96it/s]